# Exercice 4 : Ce qui est exposé, sans rien envoyer

> **En une phrase :** des moteurs comme **Shodan** scannent tout Internet en permanence. On peut donc connaître les ports et logiciels d'une cible **sans la scanner soi-même**… à condition de vérifier ce qu'ils disent.

**Le scénario.** On reprend les serveurs de uqac.ca trouvés à l'ex. 3. On veut savoir quels services ils exposent, en restant **passif**.

**La source.** *Shodan InternetDB* : une version gratuite et sans clé de Shodan. Pour une IP, elle renvoie les ports ouverts, les logiciels détectés (CPE) et les vulnérabilités (CVE) associées.

**Le plan.**
1. Retrouver les IP publiques de uqac.ca.
2. Interroger Shodan pour chaque IP.
3. À qui appartiennent vraiment ces serveurs ?
4. Les CVE : vraies vulnérabilités ou faux positifs ?

## Étape 0 : exécutez cette cellule en premier
Le notebook est **autonome** : il fonctionne tel quel sur Colab (ouvert depuis Drive) ou en local. Elle doit afficher **Prêt**.

In [1]:
import requests

def dns(nom, type_enr="A"):
    """Pose une question DNS publique (via dns.google) et renvoie la liste des réponses."""
    codes = {"A": 1, "NS": 2, "CNAME": 5, "MX": 15, "TXT": 16}
    rep = requests.get("https://dns.google/resolve", params={"name": nom, "type": type_enr}, timeout=10).json()
    return [r["data"].strip('"') for r in rep.get("Answer", []) if r["type"] == codes[type_enr]]

import time

def sous_domaines(domaine):
    """Liste les noms trouvés dans les journaux Certificate Transparency (crt.sh)."""
    for essai in range(3):                       # crt.sh est parfois lent : on réessaie
        try:
            certs = requests.get("https://crt.sh/", params={"q": "%." + domaine, "output": "json"}, timeout=90).json()
            break
        except Exception:
            print("crt.sh ne répond pas, nouvel essai…"); time.sleep(5)
    else:
        raise RuntimeError("crt.sh indisponible : réessayez dans une minute.")
    noms = set()
    for c in certs:
        for n in c["name_value"].split("\n"):
            n = n.strip().lower().removeprefix("*.")
            if n.endswith(domaine):
                noms.add(n)
    return certs, sorted(noms)

from concurrent.futures import ThreadPoolExecutor
import pandas as pd
print('Prêt')

Prêt


## Étape 1 : Retrouver les IP publiques
**Ce qu'on fait :** même collecte qu'à l'ex. 3 (certificats, puis DNS public), en parallèle pour aller vite.

In [2]:
domaine = "uqac.ca"      # <-- MODIFIABLE

_, noms = sous_domaines(domaine)
with ThreadPoolExecutor(16) as pool:
    ips = dict(zip(noms, pool.map(dns, noms)))
ips = {n: ip for n, ip in ips.items() if ip}
toutes_ips = sorted({ip for liste in ips.values() for ip in liste})
print(f"{len(ips)} noms publics → {len(toutes_ips)} adresses IP distinctes")

43 noms publics → 42 adresses IP distinctes


**👀 Ce qu'on doit voir :** ≈ **43 noms → ≈ 45 adresses IP**. *(Le nombre varie un peu d'une exécution à l'autre : certains services, comme le courriel Microsoft, changent d'IP.)*

## Étape 2 : Interroger Shodan pour chaque IP
**Ce qu'on fait :** une requête par IP vers `internetdb.shodan.io`. C'est Shodan qu'on interroge, **pas** l'UQAC.

In [3]:
def shodan(ip):
    r = requests.get(f"https://internetdb.shodan.io/{ip}", timeout=10)
    return r.json() if r.status_code == 200 else None

with ThreadPoolExecutor(8) as pool:
    infos = dict(zip(toutes_ips, pool.map(shodan, toutes_ips)))

lignes = []
for nom, liste in ips.items():
    for ip in liste:
        d = infos.get(ip) or {}
        lignes.append({"nom": nom, "ip": ip, "ports": d.get("ports", []),
                       "logiciels": [c.split(":")[2] + " " + ":".join(c.split(":")[3:]) for c in d.get("cpes", [])],
                       "nb_cve": len(d.get("vulns", []))})
expo = pd.DataFrame(lignes)
print(f"IP connues de Shodan : {sum(1 for d in infos.values() if d)} sur {len(toutes_ips)}")
expo.explode("ports")["ports"].value_counts().head(8)

IP connues de Shodan : 30 sur 42


ports
80      43
443     37
2082     7
2083     7
2052     5
2053     5
2086     5
2087     5
Name: count, dtype: int64

**👀 Ce qu'on doit voir :** ≈ **30 à 35 IP** connues de Shodan. Surtout les ports **80** et **443** (web), puis des ports 2052-2096 et 8080/8443 (typiques de Cloudflare), et quelques ports de courriel (110, 143, 993…).

## Étape 3 : À qui appartiennent vraiment ces serveurs ?
**Ce qu'on fait :** l'UQAC possède le bloc d'adresses **132.212.x.x**. Tout le reste est hébergé **chez quelqu'un d'autre**.

In [4]:
expo["hébergé par"] = expo["ip"].apply(lambda ip: "UQAC" if ip.startswith("132.212.") else "un tiers")
print(expo.groupby("nom")["hébergé par"].first().value_counts().to_string())   # un vote par nom
expo[expo["hébergé par"] == "un tiers"][["nom", "ip", "ports", "logiciels"]].head(12)

hébergé par
UQAC        22
un tiers    21


,nom,ip,ports,logiciels
0,aide.bibliotheque.uqac.ca,52.60.194.208,"[80, 443]","[jquery jquery, f5 nginx]"
1,aide.bibliotheque.uqac.ca,35.182.7.52,"[80, 443]","[jquery jquery, f5 nginx]"
3,autodiscover.uqac.ca,52.96.163.232,[80],[]
4,autodiscover.uqac.ca,40.99.236.104,[80],[]
5,autodiscover.uqac.ca,52.96.230.120,[80],[]
6,autodiscover.uqac.ca,52.96.163.248,[80],[]
7,bibliotheque.uqac.ca,52.60.177.45,"[80, 443]","[jquery jquery:3.7.1, f5 nginx, jqueryui jquer..."
8,bibliotheque.uqac.ca,35.183.221.217,"[80, 443]","[jquery jquery:3.7.1, f5 nginx, jqueryui jquer..."
9,cisd.uqac.ca,68.71.54.196,"[21, 53, 80, 110, 143, 443, 465, 587, 993, 995...","[pureftpd pure-ftpd, litespeedtech litespeed_w..."
10,classiques.uqac.ca,158.69.26.2,[],[]


**👀 Ce qu'on doit voir :** ≈ **20 noms sur 43**, presque **la moitié**, sont hébergés hors de l'UQAC : Cloudflare, Amazon (services de bibliothèque), Microsoft (courriel), un service de sondages, et même un **hébergement mutualisé** partagé avec d'autres sites commerciaux.

**❓ Question pour la classe :** un sous-domaine `xyz.uqac.ca` pointe vers un hébergement mutualisé. L'UQAC arrête de payer cet hébergeur, mais oublie de supprimer l'entrée DNS. Que peut faire un attaquant ?

*Réponse : louer le même espace chez l'hébergeur et publier **son** site sous `xyz.uqac.ca`. C'est la **prise de contrôle de sous-domaine** (subdomain takeover) : un faux site officiel, idéal pour l'hameçonnage.*

## Étape 4 : Les CVE : vraies vulnérabilités ou faux positifs ?
**Ce qu'on fait :** Shodan associe des CVE au site principal. On regarde **de quelles années** elles datent, et quel logiciel Shodan croit avoir vu.

In [5]:
ip_site = dns(domaine, "A")[0]
d = shodan(ip_site)
annees = sorted(int(c.split("-")[1]) for c in d["vulns"])
print("Logiciels détectés :", d["cpes"])
print(f"{len(d['vulns'])} CVE, publiées entre {annees[0]} et {annees[-1]}")

Logiciels détectés : ['cpe:/a:f5:big-ip', 'cpe:/a:f5:big-ip_local_traffic_manager', 'cpe:/a:wordpress:wordpress:7.1.2']
90 CVE, publiées entre 2007 et 2013


**👀 Ce qu'on doit voir :** ≈ **90 CVE**, toutes publiées entre **2007 et 2013**, pour un **WordPress 7.x**, une version de 2026. C'est impossible : ces CVE visent de vieilles extensions WordPress. Shodan a fait une correspondance trop large. **Ce sont des faux positifs.**

**❓ Question pour la classe :** que se passe-t-il si un analyste copie-colle ces 90 CVE dans un rapport ?

*Réponse : l'équipe perd du temps à corriger ce qui n'existe pas, et ne fait plus confiance aux rapports suivants. **Toujours vérifier** : la version réelle (ex. 5, en actif) et si la CVE s'applique vraiment.*

## Ce qu'il faut retenir
- Shodan a **déjà scanné** la cible : l'attaquant obtient ports et logiciels **sans se montrer**.
- La surface d'attaque d'une organisation inclut **ses fournisseurs** : près de la moitié des noms en uqac.ca sont hébergés ailleurs.
- Les outils automatiques **se trompent** : une liste de CVE n'est pas une liste de vulnérabilités. On vérifie la source et l'âge de chaque information.

**Exercice suivant :** pour vérifier, il faut parfois **toucher** la cible. On passe en reconnaissance **active**, sur une cible autorisée.